<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/GenericTransformer_Hancock_NoSmote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)




# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    np.random.seed(seed)
    random.seed(seed)


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# Load Hancock data
# ============================================================

import gdown
import json

gdown.download(
    id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",
    output="clinical_data.json",
    quiet=False
)

gdown.download(
    id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",
    output="pathological_data.json",
    quiet=False
)


with open("clinical_data.json", "r") as f:
    clinical = pd.DataFrame(json.load(f))

with open("pathological_data.json", "r") as f:
    pathology = pd.DataFrame(json.load(f))


df = clinical.merge(
    pathology,
    on="patient_id",
    how="inner"
)

print("Clinical shape:", clinical.shape)
print("Pathology shape:", pathology.shape)
print("Merged shape:", df.shape)


# ============================================================
# HPV label
# ============================================================

df = df[
    df["hpv_association_p16"].isin(
        ["positive", "negative"]
    )
].copy()

df["HPV"] = df["hpv_association_p16"].map({
    "negative": 0,
    "positive": 1
})


print("\nHPV distribution:")
print(df["HPV"].value_counts())


# ============================================================
# Hancock features
# ============================================================
#
# Do NOT include:
#   hpv_association_p16
#   HPV
#   patient_id
#
# because these contain the target or identifiers.
#
# These are the same Hancock features you have been
# using for the GPT-2 experiments.
# ============================================================

features = [

    "age_at_initial_diagnosis",
    "sex",
    "smoking_status",
    "primarily_metastasis",

    "first_treatment_intent",
    "first_treatment_modality",
    "days_to_first_treatment",

    "adjuvant_treatment_intent",
    "adjuvant_radiotherapy",
    "adjuvant_radiotherapy_modality",

    "adjuvant_systemic_therapy",
    "adjuvant_systemic_therapy_modality",
    "adjuvant_radiochemotherapy",

    "primary_tumor_site",
    "pT_stage",
    "pN_stage",
    "histologic_type",

    "number_of_positive_lymph_nodes",
    "number_of_resected_lymph_nodes",

    "perinodal_invasion",
    "lymphovascular_invasion_L",
    "vascular_invasion_V",
    "perineural_invasion_Pn",

    "resection_status",
    "infiltration_depth_in_mm"
]


# ============================================================
# Keep only required columns
# ============================================================

model_df = df[
    features + ["HPV"]
].copy()


# ============================================================
# Missing values
# ============================================================

for col in features:

    if model_df[col].dtype == object:

        mode = model_df[col].mode()

        if len(mode) > 0:
            model_df[col] = model_df[col].fillna(
                mode.iloc[0]
            )

    else:

        model_df[col] = model_df[col].fillna(
            model_df[col].median()
        )


# ============================================================
# Convert categorical variables to numerical values
# ============================================================

categorical_features = [
    col
    for col in features
    if model_df[col].dtype == object
]

numerical_features = [
    col
    for col in features
    if col not in categorical_features
]


print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)


# One-hot encode categorical variables
model_df = pd.get_dummies(
    model_df,
    columns=categorical_features,
    dtype=float
)


# ============================================================
# X and y
# ============================================================

X = model_df.drop(
    columns=["HPV"]
)

y = model_df["HPV"]


print("\nFinal feature matrix shape:", X.shape)


# ============================================================
# Train / test split
# ============================================================




X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED
)

print('train sample:',y_train.value_counts(),'\n')
print('test sample:', y_test.value_counts())



# ============================================================
# Standardisation
# ============================================================


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)





# ============================================================
# Torch Conversion
# No SMOTE
# ============================================================

X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train.to_numpy())

X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor(y_test.to_numpy())



#Model Architecture
# Transformer-based model for tabular classification
class HPVNet_Transformer(nn.Module):

    def __init__(
        self,
        num_features=78,
        d_model=32,
        nhead=4,
        num_layers=2,
        dim_feedforward=64,
        dropout=0.1,
        num_classes=2
    ):
        super().__init__()

        self.num_features = num_features
        self.d_model = d_model

        # Convert each scalar feature into a d_model-dimensional token
        self.feature_projection = nn.Linear(1, d_model)

        # Learnable embedding identifying each feature
        self.feature_embedding = nn.Parameter(
            torch.zeros(1, num_features, d_model)
        )

        # Learnable classification token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.feature_embedding, mean=0.0, std=0.02)
        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

        nn.init.xavier_uniform_(self.feature_projection.weight)
        nn.init.zeros_(self.feature_projection.bias)

    def forward(self, x):
        batch_size = x.size(0)

        # Input: [batch_size, 78]
        # Tokens: [batch_size, 78, d_model]
        x = x.unsqueeze(-1)
        x = self.feature_projection(x)

        # Distinguish each Hancock feature
        x = x + self.feature_embedding

        # Add classification token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Model interactions between tabular features
        x = self.transformer(x)

        # Use the CLS token for classification
        x = self.norm(x[:, 0])

        logits = self.classifier(x)

        return logits


model = HPVNet_Transformer(
    num_features=X_train.shape[1],
    d_model=32,
    nhead=4,
    num_layers=2,
    dim_feedforward=64,
    dropout=0.1,
    num_classes=2
)


#Defining model, and optimisation function/parameters
model = HPVNet_Transformer()
weights = torch.tensor([3.0,1.0], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
# optimizer = torch.optim.AdamW(model.parameters(),lr=0.001, weight_decay=0.01)

#Training the model
epochs = 500
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(epochs):
    model.train()
    outputs = model(X_train)
    train_loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    model.eval()

    with torch.no_grad():
        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)

    if test_loss.item() < best_val_loss:

        best_val_loss = test_loss.item()
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_model_transformer.pth")

    if (epoch+1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}",
            f"Best Epoch={best_epoch}"
        )

print("Best Test Loss =", best_val_loss)
print("Best Epoch =", best_epoch)


#Inferencing the mdoel
model.load_state_dict(torch.load("best_model_transformer.pth"))
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    probabilities = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
    predicted = torch.argmax(outputs,dim=1)

results = classification_report(y_test.numpy(),predicted.numpy(),digits=4)
print('results:',results)

# Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test.numpy(),predicted.numpy())
f1 = f1_score(y_test.numpy(), predicted.numpy())

# auc = roc_auc_score(y_test.numpy(), probabilities.numpy())
y_prob = probabilities.cpu().numpy()
if y_prob.shape[1] == 2:   # Binary classification
    auc = roc_auc_score(y_test.numpy(), y_prob[:, 1])
else:                      # Multi-class classification
    auc = roc_auc_score(
        y_test.numpy(),
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Using device: cpu


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 87.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 68.8MB/s]


Clinical shape: (763, 32)
Pathology shape: (763, 18)
Merged shape: (763, 49)

HPV distribution:
HPV
0    191
1    141
Name: count, dtype: int64

Categorical features:
['sex', 'smoking_status', 'primarily_metastasis', 'first_treatment_intent', 'first_treatment_modality', 'adjuvant_treatment_intent', 'adjuvant_radiotherapy', 'adjuvant_radiotherapy_modality', 'adjuvant_systemic_therapy', 'adjuvant_systemic_therapy_modality', 'adjuvant_radiochemotherapy', 'primary_tumor_site', 'pT_stage', 'pN_stage', 'histologic_type', 'perinodal_invasion', 'lymphovascular_invasion_L', 'vascular_invasion_V', 'perineural_invasion_Pn', 'resection_status']

Numerical features:
['age_at_initial_diagnosis', 'days_to_first_treatment', 'number_of_positive_lymph_nodes', 'number_of_resected_lymph_nodes', 'infiltration_depth_in_mm']

Final feature matrix shape: (332, 78)
train sample: HPV
0    152
1    113
Name: count, dtype: int64 

test sample: HPV
0    39
1    28
Name: count, dtype: int64


/tmp/ipykernel_763/420745524.py:336: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch 50, Train=0.4899, Test=0.4766 Best Epoch=50
Epoch 100, Train=0.3498, Test=0.3616 Best Epoch=94
Epoch 150, Train=0.3240, Test=0.4093 Best Epoch=127
Epoch 200, Train=0.2555, Test=0.5361 Best Epoch=127
Epoch 250, Train=0.2375, Test=0.5022 Best Epoch=127
Epoch 300, Train=0.2114, Test=0.5067 Best Epoch=127
Epoch 350, Train=0.3765, Test=0.3902 Best Epoch=127
Epoch 400, Train=0.2839, Test=0.4219 Best Epoch=127
Epoch 450, Train=0.2250, Test=0.5232 Best Epoch=127
Epoch 500, Train=0.2055, Test=0.5430 Best Epoch=127
Best Test Loss = 0.3428078591823578
Best Epoch = 127
results:               precision    recall  f1-score   support

           0     0.7308    0.9744    0.8352        39
           1     0.9333    0.5000    0.6512        28

    accuracy                         0.7761        67
   macro avg     0.8321    0.7372    0.7432        67
weighted avg     0.8154    0.7761    0.7583        67

Balanced Accuracy: 0.7372
F1-score:          0.6512
AUC:               0.8535
